# Build the patient-level reference standard

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


In [ ]:
import logging

import numpy as np
import pandas as pd

from smmnlp.paths import ProjectPaths

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.

## Reference standard for EndToEnd Performance


## Reference Standard


In [ ]:
readfiletrue = paths.data("Update/True/SMMNLP_testing_true_phase1.xlsx")
dataset = pd.read_excel(readfiletrue)

In [ ]:
dataset.head(5)

In [ ]:
smmfull = dataset[["PatientSSN", "Pt_index"]]
smmfull_cleaned = smmfull.drop_duplicates()

In [ ]:
len(smmfull_cleaned)

In [ ]:
readfile_name = paths.data("Update/Data/Testing/Outcome2_testing_dataset.json")
smmyes = pd.read_json(readfile_name)

In [ ]:
smmyes = smmyes.drop(columns=["NoteDate2"])

In [ ]:
smmyes["SMMDx_True_Binary"] = True

In [ ]:
smmyes["NoteDate2"] = pd.to_datetime(smmyes["NoteDate"], format="%d%b%Y")

In [ ]:
smmyes.head()

In [ ]:
len(smmyes)

In [ ]:
readfile_name = paths.data("Update/True/SMMNLP_testing_true_phase3_endtoend.xlsx")
truedata2 = pd.read_excel(readfile_name)

In [ ]:
truedata2.head(5)

In [ ]:
truedata2["HxDate2"] = pd.to_datetime(truedata2["HxDate"], format="%m/%d/%Y")

In [ ]:
smmhx = truedata2[["PatientSSN", "HxNew SMM Dx", "HxDate2", "SMM Dx Date"]]

In [ ]:
smmhx.head(5)

In [ ]:
len(smmhx)

In [ ]:
smmfull_true1 = pd.merge(smmfull_cleaned, smmyes, on=["PatientSSN"], how="left")

In [ ]:
smmfull_true2 = pd.merge(smmfull_true1, smmhx, on=["PatientSSN"], how="left")

In [ ]:
len(smmfull_true2)

In [ ]:
freq_table = smmfull_true2["SMMDx_True_Binary"].value_counts()
print(freq_table)

In [ ]:
freq_table = smmfull_true2["HxNew SMM Dx"].value_counts()
print(freq_table)

In [ ]:
smmfull_true2["True_SMMDx_Binary"] = smmfull_true2["SMMDx_True_Binary"].fillna(False)

In [ ]:
freq_table = smmfull_true2["True_SMMDx_Binary"].value_counts()
print(freq_table)

In [ ]:
smmfull_true2.head(5)

In [ ]:
smmfull_true2["True_SMMDx_Date"] = np.where(
    smmfull_true2["HxDate2"].notna(),
    smmfull_true2["HxDate2"],
    np.where(smmfull_true2["SMM Dx Date"] != "unknown", smmfull_true2["NoteDate2"], pd.NaT),
)

In [ ]:
smmfull_true2.head(5)

In [ ]:
truefinal = smmfull_true2[
    ["PatientSSN", "NoteText", "SMM Dx Date", "HxDate2", "True_SMMDx_Binary", "True_SMMDx_Date"]
]

In [ ]:
truefinal["True_SMMDx_Year"] = pd.to_datetime(truefinal["True_SMMDx_Date"]).dt.year

In [ ]:
formedian = truefinal[truefinal["True_SMMDx_Binary"] == True]

In [ ]:
formedian.head(5)

In [ ]:
median_val = truefinal["True_SMMDx_Year"].median()
q1 = truefinal["True_SMMDx_Year"].quantile(0.25)
q3 = truefinal["True_SMMDx_Year"].quantile(0.75)
iqr_val = q3 - q1
print(f"Median: {median_val}, q1: {q1}, q3: {q3}, IQR: {iqr_val}")

In [ ]:
savefile_name = paths.data("Update/True/EndToEnd_True_testing.xlsx")
truefinal.to_excel(savefile_name, engine="xlsxwriter", sheet_name="Sheet1", index=False)

In [ ]:
truefinal.head(50)